In [26]:
train_data = {
  'good': True,
  'bad': False,
  'happy': True,
  'sad': False,
  'not good': False,
  'not bad': True,
  'not happy': False,
  'not sad': True,
  'very good': True,
  'very bad': False,
  'very happy': True,
  'very sad': False,
  'i am happy': True,
  'this is good': True,
  'i am bad': False,
  'this is bad': False,
  'i am sad': False,
  'this is sad': False,
  'i am not happy': False,
  'this is not good': False,
  'i am not bad': True,
  'this is not sad': True,
  'i am very happy': True,
  'this is very good': True,
  'i am very bad': False,
  'this is very sad': False,
  'this is very happy': True,
  'i am good not bad': True,
  'this is good not bad': True,
  'i am bad not good': False,
  'i am good and happy': True,
  'this is not good and not happy': False,
  'i am not at all good': False,
  'i am not at all bad': True,
  'i am not at all happy': False,
  'this is not at all sad': True,
  'this is not at all happy': False,
  'i am good right now': True,
  'i am bad right now': False,
  'this is bad right now': False,
  'i am sad right now': False,
  'i was good earlier': True,
  'i was happy earlier': True,
  'i was bad earlier': False,
  'i was sad earlier': False,
  'i am very bad right now': False,
  'this is very good right now': True,
  'this is very sad right now': False,
  'this was bad earlier': False,
  'this was very good earlier': True,
  'this was very bad earlier': False,
  'this was very happy earlier': True,
  'this was very sad earlier': False,
  'i was good and not bad earlier': True,
  'i was not good and not happy earlier': False,
  'i am not at all bad or sad right now': True,
  'i am not at all good or happy right now': False,
  'this was not happy and not good earlier': False,
}

test_data = {
  'this is happy': True,
  'i am good': True,
  'this is not happy': False,
  'i am not good': False,
  'this is not bad': True,
  'i am not sad': True,
  'i am very good': True,
  'this is very bad': False,
  'i am very sad': False,
  'this is bad not good': False,
  'this is good and happy': True,
  'i am not good and not happy': False,
  'i am not at all sad': True,
  'this is not at all good': False,
  'this is not at all bad': True,
  'this is good right now': True,
  'this is sad right now': False,
  'this is very bad right now': False,
  'this was good earlier': True,
  'i was not happy and not good earlier': False,
}

In [27]:

vocab = list(set([w for text in train_data.keys() for w in text.split(' ')]))
vocab_size = len(vocab)

word2idx = { w:i for i,w in enumerate(vocab)}
idx2word = { i:w for i,w in enumerate(vocab)}


In [28]:
import numpy as np

def transInput(text):
    inputs = []
    for w in text.split(' '):
        v = np.zeros((vocab_size,1))
        v[word2idx[w]] = 1
        inputs.append(v)

    return inputs

def softmax(x):
    return np.exp(x) / sum(np.exp(x))


In [29]:
import random

class RNN:
    def __init__(self, input_size, output_size, hidden_size = 64):
        self.whh = np.random.randn(hidden_size, hidden_size)
        self.whh /= np.max(self.whh)

        self.wxh = np.random.randn(hidden_size, input_size)
        self.wxh /= np.max(self.wxh)

        self.why = np.random.randn(output_size, hidden_size)
        self.why /= np.max(self.why)

        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = np.zeros((self.whh.shape[0], 1))

        self.last_inputs = inputs
        self.last_hs = { 0: h }

        for i,x in enumerate(inputs):
            h = np.tanh(self.wxh @ x + self.whh @ h + self.bh)
            self.last_hs[i + 1] = h

        y = self.why @ h + self.by

        return y,h
    
    def backprop(self, d_L_d_y, lr = 0.01):
        n = len(self.last_inputs)

        d_L_d_why = d_L_d_y @ self.last_hs[n].T
        d_L_d_by = d_L_d_y

        d_L_d_whh = np.zeros(self.whh.shape)
        d_L_d_wxh = np.zeros(self.wxh.shape)
        d_L_d_bh = np.zeros(self.bh.shape)

        d_L_d_ht = self.why.T @ d_L_d_y

        for t in reversed(range(n)):
            temp = (1 - self.last_hs[t+1] ** 2) * d_L_d_ht

            d_L_d_bh += temp
            d_L_d_whh += temp @ self.last_hs[t].T
            d_L_d_wxh += temp @ self.last_inputs[t].T

            d_L_d_ht = self.whh @ temp

        for d in [d_L_d_whh,d_L_d_why,d_L_d_wxh,d_L_d_by,d_L_d_bh]:
            np.clip(d,-1,1,out=d)

        self.whh -= lr * d_L_d_whh
        self.why -= lr * d_L_d_why
        self.wxh -= lr * d_L_d_wxh
        self.bh -= lr * d_L_d_bh
        self.by -= lr * d_L_d_by

    def process(self, data, bp = True):
        items = list(data.items())
        random.shuffle(items)

        loss = 0
        num_correct = 0

        for x,y in items:
            inputs = transInput(x)
            target = int(y)

            out, _ = self.forward(inputs)
            probs = softmax(out)

            loss += -np.log(probs[target]).item()
            num_correct += int(np.argmax(probs) == target)

            if bp:
                d_L_d_y = probs
                d_L_d_y[target] -= 1

                self.backprop(d_L_d_y)

        return loss / len(data), num_correct / len(data)

    def train(self, train_data, test_data, epoches = 1000):
        for epoch in range(epoches):
            train_loss, train_acc = self.process(train_data)

            if(epoch % 100 == 99):
                print(f"--- epoch {epoch+1}:")
                print("Train:\tLoss %.3f | Accuracy: %.3f" % (train_loss, train_acc))

                test_loss, test_acc = self.process(test_data, bp = False)
                print("Test:\tLoss %.3f | Accuracy: %.3f" % (test_loss, test_acc))




In [30]:

rnn = RNN(vocab_size, 2)
rnn.train(train_data, test_data)

--- epoch 100:
Train:	Loss 0.058 | Accuracy: 1.000
Test:	Loss 0.506 | Accuracy: 0.700
--- epoch 200:
Train:	Loss 0.012 | Accuracy: 1.000
Test:	Loss 0.228 | Accuracy: 0.900
--- epoch 300:
Train:	Loss 0.006 | Accuracy: 1.000
Test:	Loss 0.131 | Accuracy: 1.000
--- epoch 400:
Train:	Loss 0.003 | Accuracy: 1.000
Test:	Loss 0.102 | Accuracy: 1.000
--- epoch 500:
Train:	Loss 0.003 | Accuracy: 1.000
Test:	Loss 0.074 | Accuracy: 1.000
--- epoch 600:
Train:	Loss 0.002 | Accuracy: 1.000
Test:	Loss 0.098 | Accuracy: 0.950
--- epoch 700:
Train:	Loss 0.002 | Accuracy: 1.000
Test:	Loss 0.110 | Accuracy: 0.950
--- epoch 800:
Train:	Loss 0.001 | Accuracy: 1.000
Test:	Loss 0.082 | Accuracy: 0.950
--- epoch 900:
Train:	Loss 0.001 | Accuracy: 1.000
Test:	Loss 0.063 | Accuracy: 1.000
--- epoch 1000:
Train:	Loss 0.001 | Accuracy: 1.000
Test:	Loss 0.051 | Accuracy: 1.000
